# Matrix Profile Study Summary (Fast Development Notebook)

This short notebook is the fast development version of the BTCUSDT Matrix Profile study.
It keeps the core thesis logic, but it is organized to give fast feedback first:

- `quick`: about 14 days of 1-minute data, cached univariate study, lightweight plots
- `medium`: about 90 days, optional cached multivariate study
- `full`: all available data, intended for cached reruns or long CPU sessions

The main focus is to extract useful motif pairs quickly, reuse cached matrix profiles, and
produce intuitive charts that are easy to discuss with a supervisor.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import time
from pathlib import Path
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import stumpy

from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.rcParams["figure.dpi"] = 130


In [ ]:
ASSET = "BTCUSDT"

RUN_MODE = "quick"
FORCE_RECOMPUTE = False
FEATURE = "log_return"  # change to "close" for raw-price motifs

RUN_CONFIGS = {
    "quick": {
        "start": None,
        "end": None,
        "m_intraday": 60,
        "m_daily": 1440,
        "top_k": 3,
        "run_raw_close": True,
        "run_univariate": True,
        "run_multivariate": False,
        "run_event_matching": False,
        "run_heavy_plots": False,
    },
    "medium": {
        "start": None,
        "end": None,
        "m_intraday": 60,
        "m_daily": 1440,
        "top_k": 5,
        "run_raw_close": True,
        "run_univariate": True,
        "run_multivariate": True,
        "run_event_matching": False,
        "run_heavy_plots": True,
    },
    "full": {
        "start": None,
        "end": None,
        "m_intraday": 60,
        "m_daily": 1440,
        "top_k": 10,
        "run_raw_close": True,
        "run_univariate": True,
        "run_multivariate": True,
        "run_event_matching": True,
        "run_heavy_plots": True,
    },
}

AUTO_SLICE_DAYS = {
    "quick": 14,
    "medium": 90,
    "full": None,
}

RUNTIME_HINTS = {
    "quick": "Designed to finish quickly on CPU and reuse caches when they exist.",
    "medium": "Reasonable development run for a longer slice. Multivariate MSTUMP can still take time.",
    "full": "Full-history run. Best used when caches already exist or long execution is acceptable.",
}

if RUN_MODE not in RUN_CONFIGS:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

CFG = RUN_CONFIGS[RUN_MODE].copy()

display(pd.DataFrame([{
    "run_mode": RUN_MODE,
    "force_recompute": FORCE_RECOMPUTE,
    "feature": FEATURE,
    **CFG,
}]))


## Path Setup

The notebook should run from either the repository root or the notebook folder.
All important output folders are created automatically.


In [ ]:
def find_project_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate the project root from {start}")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "MP_Study_Summary"

DATA_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "crypto" / "1min" / f"{ASSET}_1m_processed.parquet",
    PROJECT_ROOT / "data" / "processed" / f"{ASSET}_1m_processed.parquet",
    PROJECT_ROOT / "data" / "processed" / f"{ASSET}.parquet",
]

EXTERNAL_EVENT_CANDIDATES = [
    PROJECT_ROOT / "data" / "external" / f"{ASSET}_events.csv",
    PROJECT_ROOT / "data" / "external" / f"{ASSET}_2025_events.csv",
    PROJECT_ROOT / "data" / "external" / "crypto_2025_events.csv",
    PROJECT_ROOT / "data" / "external" / "external_events.csv",
]

CACHE_DIR = PROJECT_ROOT / "data" / "processed" / "matrix_profiles"
RESULTS_DIR = PROJECT_ROOT / "reports" / "results" / "MP_Study_Summary_short"
PLOTS_DIR = RESULTS_DIR / "plots"

for directory in [CACHE_DIR, RESULTS_DIR, PLOTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CACHE_DIR:", CACHE_DIR)
print("RESULTS_DIR:", RESULTS_DIR)


## Load Processed BTCUSDT Parquet

This section loads the processed one-minute BTCUSDT file, converts timestamps to UTC, and
enforces chronological ordering.


In [ ]:
def find_existing_file(candidates: List[Path]) -> Path:
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("No BTCUSDT processed file found. Checked:\n" + "\n".join(str(p) for p in candidates))


def load_processed_data(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        df_loaded = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df_loaded = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    if "timestamp" not in df_loaded.columns:
        raise ValueError("The processed dataset must contain a 'timestamp' column.")

    df_loaded = df_loaded.copy()
    df_loaded["timestamp"] = pd.to_datetime(df_loaded["timestamp"], utc=True, errors="coerce")
    df_loaded = df_loaded.dropna(subset=["timestamp"]).sort_values("timestamp")
    df_loaded = df_loaded.drop_duplicates("timestamp", keep="last").reset_index(drop=True)

    numeric_cols = [col for col in df_loaded.columns if col != "timestamp"]
    for col in numeric_cols:
        df_loaded[col] = pd.to_numeric(df_loaded[col], errors="coerce")

    return df_loaded


data_path = find_existing_file(DATA_CANDIDATES)
df_full = load_processed_data(data_path)

print("Using dataset:", data_path)
print(f"Loaded rows: {len(df_full):,}")
print("Full dataset range:", df_full["timestamp"].min(), "to", df_full["timestamp"].max())
print("Columns:", df_full.columns.tolist())

display(df_full.head(3))


## Apply Time Slice

The short notebook should be easy to rerun. The helper below resolves a practical slice for
each run mode and prints a runtime-friendly summary before any Matrix Profile work begins.


In [ ]:
def to_utc_timestamp(value: Optional[object]) -> Optional[pd.Timestamp]:
    if value is None:
        return None
    return pd.to_datetime(value, utc=True)


def resolve_slice_bounds(df_in: pd.DataFrame, cfg: Dict[str, object], run_mode: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    if df_in.empty:
        raise ValueError("The full dataframe is empty.")

    data_start = df_in["timestamp"].min()
    data_end_exclusive = df_in["timestamp"].max() + pd.Timedelta(minutes=1)

    configured_start = to_utc_timestamp(cfg.get("start"))
    configured_end = to_utc_timestamp(cfg.get("end"))

    if run_mode == "full":
        start = configured_start or data_start
        end = configured_end or data_end_exclusive
        return start, end

    end = configured_end or data_end_exclusive
    auto_days = AUTO_SLICE_DAYS[run_mode]
    start = configured_start or (end - pd.Timedelta(days=auto_days))

    start = max(start, data_start)
    end = min(end, data_end_exclusive)

    if start >= end:
        raise ValueError("Resolved slice bounds are invalid. Check the configured start and end values.")

    return start, end


def apply_time_slice(
    df_in: pd.DataFrame,
    timestamp_col: str = "timestamp",
    start: Optional[pd.Timestamp] = None,
    end: Optional[pd.Timestamp] = None,
) -> pd.DataFrame:
    if timestamp_col not in df_in.columns:
        raise KeyError(f"Missing timestamp column: {timestamp_col}")

    df_out = df_in.copy()
    df_out[timestamp_col] = pd.to_datetime(df_out[timestamp_col], utc=True, errors="coerce")
    df_out = df_out.dropna(subset=[timestamp_col]).sort_values(timestamp_col)

    if start is not None:
        df_out = df_out[df_out[timestamp_col] >= start]
    if end is not None:
        df_out = df_out[df_out[timestamp_col] < end]

    return df_out.reset_index(drop=True)


slice_start, slice_end = resolve_slice_bounds(df_full, CFG, RUN_MODE)
df = apply_time_slice(df_full, start=slice_start, end=slice_end)

if df.empty:
    raise ValueError(f"No rows remain after applying RUN_MODE={RUN_MODE} with start={slice_start} and end={slice_end}.")

slice_summary = pd.DataFrame([{
    "run_mode": RUN_MODE,
    "rows_used": len(df),
    "start": df["timestamp"].min(),
    "end": df["timestamp"].max(),
    "approx_days": round((df["timestamp"].max() - df["timestamp"].min()) / pd.Timedelta(days=1), 2),
    "runtime_hint": RUNTIME_HINTS[RUN_MODE],
}])

print("Sliced dataframe shape:", df.shape)
print("Slice start:", df["timestamp"].min())
print("Slice end:", df["timestamp"].max())
print("Runtime summary:", RUNTIME_HINTS[RUN_MODE])
display(slice_summary)


## Feature Checks And Feature Creation

The thesis study uses a small set of engineered one-minute features. If some columns are not
already present in the processed parquet, the notebook creates them here.


In [ ]:
ROLLING_VOL_WINDOW_MINUTES = 60
VOLUME_ZSCORE_WINDOW_MINUTES = 1440


def add_engineered_features(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()

    required_ohlcv = {"open", "high", "low", "close", "volume"}
    missing_ohlcv = sorted(required_ohlcv - set(df_out.columns))
    if missing_ohlcv:
        raise ValueError(f"Missing required OHLCV columns: {missing_ohlcv}")

    if "log_return" not in df_out.columns:
        df_out["log_return"] = np.log(df_out["close"]).diff()

    if "pct_return" not in df_out.columns:
        df_out["pct_return"] = df_out["close"].pct_change()

    if "hl_range" not in df_out.columns:
        df_out["hl_range"] = (df_out["high"] - df_out["low"]) / df_out["close"].replace(0, np.nan)

    if "rolling_volatility" not in df_out.columns:
        preferred_vol_cols = [
            "realized_volatility_60m",
            "volatility_60m",
            "volatility_30m",
            "volatility_240m",
        ]
        source_col = next((col for col in preferred_vol_cols if col in df_out.columns), None)
        if source_col is not None:
            df_out["rolling_volatility"] = df_out[source_col]
            df_out["rolling_volatility_source"] = source_col
        else:
            df_out["rolling_volatility"] = df_out["log_return"].rolling(
                ROLLING_VOL_WINDOW_MINUTES,
                min_periods=ROLLING_VOL_WINDOW_MINUTES,
            ).std()
            df_out["rolling_volatility_source"] = f"log_return_rolling_{ROLLING_VOL_WINDOW_MINUTES}m_std"

    if "volume_zscore" not in df_out.columns:
        volume_mean = df_out["volume"].rolling(
            VOLUME_ZSCORE_WINDOW_MINUTES,
            min_periods=VOLUME_ZSCORE_WINDOW_MINUTES,
        ).mean()
        volume_std = df_out["volume"].rolling(
            VOLUME_ZSCORE_WINDOW_MINUTES,
            min_periods=VOLUME_ZSCORE_WINDOW_MINUTES,
        ).std()
        df_out["volume_zscore"] = (df_out["volume"] - volume_mean) / volume_std.replace(0, np.nan)

    engineered_cols = ["log_return", "rolling_volatility", "hl_range", "volume_zscore"]
    for col in engineered_cols:
        df_out[col] = pd.to_numeric(df_out[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

    return df_out


df = add_engineered_features(df)

if FEATURE not in df.columns:
    raise KeyError(f"Selected FEATURE='{FEATURE}' is not available after feature engineering.")

MULTIVARIATE_FEATURES = [col for col in ["log_return", "rolling_volatility", "hl_range", "volume_zscore"] if col in df.columns]

feature_check_df = pd.DataFrame([
    {
        "feature": col,
        "non_null_rows": int(df[col].replace([np.inf, -np.inf], np.nan).notna().sum()),
        "missing_fraction": float(df[col].replace([np.inf, -np.inf], np.nan).isna().mean()),
    }
    for col in sorted(set(["close", FEATURE] + MULTIVARIATE_FEATURES))
    if col in df.columns
])

print("Available multivariate features:", MULTIVARIATE_FEATURES)
if "rolling_volatility_source" in df.columns and df["rolling_volatility_source"].notna().any():
    print("rolling_volatility source:", df["rolling_volatility_source"].dropna().iloc[0])

display(feature_check_df)


## Cache Helpers

Cached Matrix Profiles are stored under `data/processed/matrix_profiles/`. The file names
include the asset, method, feature signature, window size, and run mode.


In [ ]:
def safe_stem(value: object) -> str:
    text = str(value)
    cleaned = "".join(ch if ch.isalnum() or ch in {"-", "_"} else "_" for ch in text)
    return cleaned.strip("_") or "unnamed"


def build_cache_path(asset: str, method: str, feature_name: str, m: int, run_mode: str) -> Path:
    return CACHE_DIR / f"{safe_stem(asset)}_{safe_stem(method)}_{safe_stem(feature_name)}_m{int(m)}_{safe_stem(run_mode)}.npz"


def save_npz_cache(path: Path, **arrays) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **arrays)


def load_npz_cache(path: Path) -> Optional[Dict[str, np.ndarray]]:
    if not path.exists():
        return None
    with np.load(path, allow_pickle=True) as cached:
        return {name: cached[name] for name in cached.files}


def to_cache_datetime_ns(timestamps: pd.Series) -> np.ndarray:
    ts = pd.to_datetime(timestamps, utc=True)
    return pd.DatetimeIndex(ts).asi8


def from_cache_datetime_ns(values: np.ndarray) -> pd.Series:
    raw = np.asarray(values, dtype="int64")
    unit = "ns" if np.nanmax(np.abs(raw)) > 10**15 else "ms"
    return pd.Series(pd.to_datetime(raw, unit=unit, utc=True), name="timestamp")


## Matrix Profile Helpers

The helper functions below clean the input series, extract top non-overlapping motifs, and
manage cached univariate and multivariate Matrix Profile calculations.


In [ ]:
MOTIF_COLUMNS = [
    "rank",
    "motif_idx",
    "neighbor_idx",
    "motif_start",
    "motif_end",
    "neighbor_start",
    "neighbor_end",
    "profile_value",
]


def empty_motif_table() -> pd.DataFrame:
    return pd.DataFrame(columns=MOTIF_COLUMNS)


def clean_numeric_series(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)


def prepare_univariate_series(df_in: pd.DataFrame, feature: str) -> pd.DataFrame:
    if feature not in df_in.columns:
        raise KeyError(f"Missing feature column: {feature}")
    prepared = df_in[["timestamp", feature]].copy()
    prepared[feature] = clean_numeric_series(prepared[feature])
    prepared = prepared.dropna(subset=[feature]).reset_index(drop=True)
    return prepared


def prepare_multivariate_frame(df_in: pd.DataFrame, features: List[str]) -> pd.DataFrame:
    missing = [feature for feature in features if feature not in df_in.columns]
    if missing:
        raise KeyError(f"Missing multivariate features: {missing}")
    prepared = df_in[["timestamp"] + features].copy()
    for feature in features:
        prepared[feature] = clean_numeric_series(prepared[feature])
    prepared = prepared.dropna(subset=features).reset_index(drop=True)
    return prepared


def z_normalize(values: np.ndarray) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    mean = np.nanmean(arr)
    std = np.nanstd(arr)
    if np.isnan(std) or std == 0:
        return np.zeros_like(arr, dtype=float)
    return (arr - mean) / std


def cache_matches_slice(
    cached: Dict[str, np.ndarray],
    expected_start: str,
    expected_end: str,
    expected_rows: int,
    expected_feature_signature: str,
) -> bool:
    cached_start = str(np.atleast_1d(cached.get("slice_start", np.array([""])))[0])
    cached_end = str(np.atleast_1d(cached.get("slice_end", np.array([""])))[0])
    cached_rows = int(np.atleast_1d(cached.get("series_length", cached.get("rows_used", np.array([-1]))))[0])

    if "feature" in cached:
        cached_feature_signature = str(np.atleast_1d(cached["feature"])[0])
    elif "features" in cached:
        cached_feature_signature = "__".join(np.asarray(cached["features"], dtype=str).tolist())
    else:
        cached_feature_signature = ""

    return (
        cached_start == expected_start
        and cached_end == expected_end
        and cached_rows == int(expected_rows)
        and cached_feature_signature == expected_feature_signature
    )


def extract_top_k_motifs(
    mp: np.ndarray,
    mpi: np.ndarray,
    timestamps: pd.Series,
    m: int,
    top_k: int,
) -> pd.DataFrame:
    mp = np.asarray(mp, dtype=float)
    mpi = np.asarray(mpi, dtype=int)
    ts = pd.Series(pd.to_datetime(timestamps, utc=True)).reset_index(drop=True)

    if len(ts) < m or len(mp) == 0:
        return empty_motif_table()

    chosen_rows = []
    occupied_starts: List[int] = []

    for motif_idx in np.argsort(mp):
        if len(chosen_rows) >= top_k:
            break

        if motif_idx >= len(mp) or not np.isfinite(mp[motif_idx]):
            continue

        neighbor_idx = int(mpi[motif_idx]) if motif_idx < len(mpi) else -1
        if neighbor_idx < 0:
            continue
        if motif_idx + m > len(ts) or neighbor_idx + m > len(ts):
            continue
        if abs(int(motif_idx) - neighbor_idx) < m:
            continue

        candidate_starts = [int(motif_idx), neighbor_idx]
        overlaps_existing = any(
            abs(candidate_start - existing_start) < m
            for candidate_start in candidate_starts
            for existing_start in occupied_starts
        )
        if overlaps_existing:
            continue

        chosen_rows.append({
            "rank": len(chosen_rows) + 1,
            "motif_idx": int(motif_idx),
            "neighbor_idx": int(neighbor_idx),
            "motif_start": ts.iloc[motif_idx],
            "motif_end": ts.iloc[motif_idx + m - 1],
            "neighbor_start": ts.iloc[neighbor_idx],
            "neighbor_end": ts.iloc[neighbor_idx + m - 1],
            "profile_value": float(mp[motif_idx]),
        })
        occupied_starts.extend(candidate_starts)

    if not chosen_rows:
        return empty_motif_table()

    return pd.DataFrame(chosen_rows, columns=MOTIF_COLUMNS)


def build_occurrence_table(motif_df: pd.DataFrame, representation: str) -> pd.DataFrame:
    if motif_df.empty:
        return pd.DataFrame(columns=[
            "representation",
            "rank",
            "occurrence_type",
            "window_size",
            "start",
            "end",
            "profile_value",
        ])

    rows = []
    window_size = int((motif_df["motif_end"].iloc[0] - motif_df["motif_start"].iloc[0]) / pd.Timedelta(minutes=1) + 1)
    for _, row in motif_df.iterrows():
        rows.append({
            "representation": representation,
            "rank": int(row["rank"]),
            "occurrence_type": "motif",
            "window_size": window_size,
            "start": row["motif_start"],
            "end": row["motif_end"],
            "profile_value": float(row["profile_value"]),
        })
        rows.append({
            "representation": representation,
            "rank": int(row["rank"]),
            "occurrence_type": "neighbor",
            "window_size": window_size,
            "start": row["neighbor_start"],
            "end": row["neighbor_end"],
            "profile_value": float(row["profile_value"]),
        })
    return pd.DataFrame(rows)


def compute_cached_univariate_stump(
    df_in: pd.DataFrame,
    feature: str,
    m: int,
    asset: str,
    run_mode: str,
    force_recompute: bool = False,
) -> Dict[str, object]:
    series_df = prepare_univariate_series(df_in, feature)
    series = series_df[feature].to_numpy(dtype=float)
    timestamps = series_df["timestamp"].reset_index(drop=True)

    if len(series) <= m:
        raise ValueError(f"Not enough rows for feature='{feature}' and m={m}. Need more than {m} points.")

    cache_path = build_cache_path(asset, "stump", feature, m, run_mode)
    slice_start = timestamps.iloc[0].isoformat()
    slice_end = timestamps.iloc[-1].isoformat()
    expected_signature = feature

    mp = np.array([], dtype=float)
    mpi = np.array([], dtype=int)
    cache_used = False
    elapsed_seconds = 0.0

    cached = None if force_recompute else load_npz_cache(cache_path)
    if cached is not None and {"mp", "mpi"} <= set(cached.keys()):
        if cache_matches_slice(cached, slice_start, slice_end, len(series), expected_signature):
            mp = np.asarray(cached["mp"], dtype=float)
            mpi = np.asarray(cached["mpi"], dtype=int)
            if "series" in cached and len(cached["series"]) == len(series):
                series = np.asarray(cached["series"], dtype=float)
            if "timestamps_ns" in cached and len(cached["timestamps_ns"]) == len(timestamps):
                timestamps = from_cache_datetime_ns(cached["timestamps_ns"]).reset_index(drop=True)
            cache_used = True
            print(f"Loaded cached matrix profile: {cache_path.name}")
        else:
            print(f"Cache metadata changed for {cache_path.name}. Recomputing.")

    if not cache_used:
        start_time = time.perf_counter()
        mp_raw = stumpy.stump(series, m=m)
        elapsed_seconds = time.perf_counter() - start_time

        mp = np.asarray(mp_raw[:, 0], dtype=float)
        mpi = np.asarray(mp_raw[:, 1], dtype=int)

        save_npz_cache(
            cache_path,
            mp=mp,
            mpi=mpi,
            series=series,
            timestamps_ns=to_cache_datetime_ns(timestamps),
            m=np.array([m]),
            feature=np.array([feature]),
            run_mode=np.array([run_mode]),
            slice_start=np.array([slice_start]),
            slice_end=np.array([slice_end]),
            series_length=np.array([len(series)]),
        )
        print(f"Computed matrix profile and cached it: {cache_path.name}")
        print(f"Elapsed time: {elapsed_seconds:.2f} seconds")

    print("Cache status:", "loaded" if cache_used else "computed")
    print("Series length:", len(series))
    print("Matrix profile length:", len(mp))
    print("Finite MP values:", int(np.isfinite(mp).sum()))

    return {
        "feature": feature,
        "m": int(m),
        "cache_path": cache_path,
        "cache_used": bool(cache_used),
        "elapsed_seconds": float(elapsed_seconds),
        "series": series,
        "timestamps": timestamps,
        "series_df": pd.DataFrame({"timestamp": timestamps, feature: series}),
        "mp": mp,
        "mpi": mpi,
    }


def compute_cached_mstump(
    df_in: pd.DataFrame,
    features: List[str],
    m: int,
    asset: str,
    run_mode: str,
    force_recompute: bool = False,
) -> Dict[str, object]:
    if len(features) < 2:
        raise ValueError("Need at least two features for multivariate MSTUMP.")

    multi_df = prepare_multivariate_frame(df_in, features)
    if len(multi_df) <= m:
        raise ValueError(f"Not enough multivariate rows for m={m}.")

    X = multi_df[features].to_numpy(dtype=float).T
    timestamps = multi_df["timestamp"].reset_index(drop=True)

    feature_signature = "__".join(features)
    cache_path = build_cache_path(asset, "mstump", feature_signature, m, run_mode)
    slice_start = timestamps.iloc[0].isoformat()
    slice_end = timestamps.iloc[-1].isoformat()

    mps = np.empty((0, 0), dtype=float)
    indices = np.empty((0, 0), dtype=int)
    cache_used = False
    elapsed_seconds = 0.0

    cached = None if force_recompute else load_npz_cache(cache_path)
    if cached is not None and {"mps", "indices"} <= set(cached.keys()):
        if cache_matches_slice(cached, slice_start, slice_end, X.shape[1], feature_signature):
            mps = np.asarray(cached["mps"], dtype=float)
            indices = np.asarray(cached["indices"], dtype=int)
            if "X" in cached and cached["X"].shape == X.shape:
                X = np.asarray(cached["X"], dtype=float)
            if "timestamps_ns" in cached and len(cached["timestamps_ns"]) == len(timestamps):
                timestamps = from_cache_datetime_ns(cached["timestamps_ns"]).reset_index(drop=True)
            cache_used = True
            print(f"Loaded cached MSTUMP profile: {cache_path.name}")
        else:
            print(f"Cache metadata changed for {cache_path.name}. Recomputing.")

    if not cache_used:
        start_time = time.perf_counter()
        mps, indices = stumpy.mstump(X, m=m)
        elapsed_seconds = time.perf_counter() - start_time

        mps = np.asarray(mps, dtype=float)
        indices = np.asarray(indices, dtype=int)

        save_npz_cache(
            cache_path,
            mps=mps,
            indices=indices,
            X=X,
            timestamps_ns=to_cache_datetime_ns(timestamps),
            features=np.asarray(features, dtype=str),
            m=np.array([m]),
            run_mode=np.array([run_mode]),
            slice_start=np.array([slice_start]),
            slice_end=np.array([slice_end]),
            rows_used=np.array([X.shape[1]]),
        )
        print(f"Computed MSTUMP and cached it: {cache_path.name}")
        print(f"Elapsed time: {elapsed_seconds:.2f} seconds")

    full_profile = np.asarray(mps[-1], dtype=float)
    full_indices = np.asarray(indices[-1], dtype=int)

    print("Cache status:", "loaded" if cache_used else "computed")
    print("Multivariate rows used:", X.shape[1])
    print("Multivariate profile length:", len(full_profile))
    print("Finite full-dimensional MP values:", int(np.isfinite(full_profile).sum()))

    return {
        "features": features,
        "m": int(m),
        "cache_path": cache_path,
        "cache_used": bool(cache_used),
        "elapsed_seconds": float(elapsed_seconds),
        "multi_df": multi_df,
        "timestamps": timestamps,
        "X": X,
        "mps": mps,
        "indices": indices,
        "full_profile": full_profile,
        "full_indices": full_indices,
    }


## Cached Univariate STUMP

The fast notebook computes one main univariate Matrix Profile using `stumpy.stump`.
By default this uses `FEATURE = "log_return"`, but you can change it to `FEATURE = "close"`
if you want raw price motifs instead. An optional raw-close baseline is also available.


In [ ]:
analysis_results: Dict[str, Dict[str, object]] = {}
saved_plot_paths: List[Path] = []
runtime_rows: List[Dict[str, object]] = []
multivariate_result: Optional[Dict[str, object]] = None
multivariate_motifs_df = empty_motif_table()
events_df = pd.DataFrame()
event_matches = pd.DataFrame()

requested_univariate_features: List[str] = []
if CFG["run_univariate"]:
    requested_univariate_features.append(FEATURE)
if CFG["run_raw_close"] and "close" in df.columns and FEATURE != "close":
    requested_univariate_features.append("close")

if not requested_univariate_features:
    raise ValueError("No univariate study is enabled. Set run_univariate=True or run_raw_close=True.")

for current_feature in requested_univariate_features:
    print(f"\nRunning cached STUMP for feature='{current_feature}' with m={CFG['m_intraday']}")
    analysis_results[current_feature] = compute_cached_univariate_stump(
        df_in=df,
        feature=current_feature,
        m=CFG["m_intraday"],
        asset=ASSET,
        run_mode=RUN_MODE,
        force_recompute=FORCE_RECOMPUTE,
    )

primary_result = analysis_results.get(FEATURE) or analysis_results.get("close")
if primary_result is None:
    raise ValueError("Could not select a primary univariate result.")

raw_close_result = analysis_results.get("close") if FEATURE != "close" else None


## Motif Extraction

This step extracts the top non-overlapping motif pairs from the cached Matrix Profile results.
The lowest finite profile values are treated as the strongest candidate motifs.


In [ ]:
for current_feature, result in analysis_results.items():
    motif_table = extract_top_k_motifs(
        mp=result["mp"],
        mpi=result["mpi"],
        timestamps=result["timestamps"],
        m=result["m"],
        top_k=CFG["top_k"],
    )
    result["motifs_df"] = motif_table

    runtime_rows.append({
        "run_mode": RUN_MODE,
        "rows_used": int(len(result["series"])),
        "feature": current_feature,
        "window_size": int(result["m"]),
        "top_k": int(CFG["top_k"]),
        "cache_used": bool(result["cache_used"]),
        "elapsed_seconds": round(float(result["elapsed_seconds"]), 4),
    })

    label = "primary study" if current_feature == primary_result["feature"] else "raw-close baseline"
    print(f"\nTop motifs for {current_feature} ({label}):")
    display(motif_table)

motifs_df = primary_result["motifs_df"].copy()
if motifs_df.empty:
    print("No valid motifs were extracted for the primary feature.")
else:
    print("Primary motif table used for plots and exports:")
    display(motifs_df)


## Intuitive Visualizations

These plots are designed to be lightweight and easy to explain in thesis discussions:

1. the selected windows highlighted on a line chart
2. the Matrix Profile curve with motif locations marked
3. z-normalized motif overlays that show why two windows were matched


In [ ]:
def build_plot_path(feature: str, suffix: str) -> Path:
    return PLOTS_DIR / f"{safe_stem(ASSET)}_{safe_stem(feature)}_{safe_stem(suffix)}_{RUN_MODE}.png"


def save_and_show_figure(fig: plt.Figure, path: Path) -> None:
    fig.savefig(path, dpi=200, bbox_inches="tight")
    saved_plot_paths.append(path)
    plt.show()
    plt.close(fig)


def plot_detected_motif_windows(
    df_signal: pd.DataFrame,
    value_col: str,
    motif_table: pd.DataFrame,
    study_label: str,
    save_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(15, 4))
    ax.plot(df_signal["timestamp"], df_signal[value_col], linewidth=0.9, color="#1f3b73")

    colors = ["#6bbf59", "#f0a202", "#7b5ea7"]
    for idx, (_, row) in enumerate(motif_table.head(3).iterrows()):
        color = colors[idx % len(colors)]
        ax.axvspan(row["motif_start"], row["motif_end"], color=color, alpha=0.22, label=f"Motif {int(row['rank'])}")
        ax.axvspan(
            row["neighbor_start"],
            row["neighbor_end"],
            color=color,
            alpha=0.10,
            hatch="//",
            label=f"Neighbor {int(row['rank'])}",
        )

    ax.set_title(f"Detected Similar Motif Windows | {study_label}")
    ax.set_ylabel(value_col)
    ax.set_xlabel("Timestamp")
    handles, labels = ax.get_legend_handles_labels()
    dedup = dict(zip(labels, handles))
    ax.legend(dedup.values(), dedup.keys(), ncol=3, loc="upper left")
    fig.tight_layout()
    save_and_show_figure(fig, save_path)


def plot_matrix_profile_curve(
    result: Dict[str, object],
    motif_table: pd.DataFrame,
    save_path: Path,
) -> None:
    mp = np.asarray(result["mp"], dtype=float)
    mp_timestamps = pd.Series(result["timestamps"]).iloc[:len(mp)]

    fig, ax = plt.subplots(figsize=(15, 4))
    ax.plot(mp_timestamps, mp, linewidth=0.9, color="#2f4858")

    top_motifs = motif_table.head(3)
    if not top_motifs.empty:
        motif_idx = top_motifs["motif_idx"].to_numpy(dtype=int)
        neighbor_idx = top_motifs["neighbor_idx"].to_numpy(dtype=int)
        ax.scatter(mp_timestamps.iloc[motif_idx], mp[motif_idx], color="green", s=35, label="Motif index", zorder=3)
        ax.scatter(mp_timestamps.iloc[neighbor_idx], mp[neighbor_idx], color="orange", s=35, label="Nearest neighbor", zorder=3)

    ax.set_title(f"Matrix Profile Curve | {result['feature']} | m={result['m']}")
    ax.set_ylabel("Matrix Profile distance")
    ax.set_xlabel("Timestamp")
    ax.legend(loc="upper right")
    fig.tight_layout()
    save_and_show_figure(fig, save_path)


def plot_normalized_motif_overlay(
    result: Dict[str, object],
    motif_row: pd.Series,
    save_path: Path,
) -> None:
    feature = result["feature"]
    m = int(result["m"])
    motif_idx = int(motif_row["motif_idx"])
    neighbor_idx = int(motif_row["neighbor_idx"])

    seq_a = z_normalize(result["series"][motif_idx: motif_idx + m])
    seq_b = z_normalize(result["series"][neighbor_idx: neighbor_idx + m])

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(seq_a, linewidth=1.5, label=f"Motif window | rank {int(motif_row['rank'])}")
    ax.plot(seq_b, linewidth=1.5, linestyle="--", label="Nearest-neighbor window")
    ax.set_title(
        f"Normalized Motif Overlay | {feature} | rank={int(motif_row['rank'])} | MP={motif_row['profile_value']:.4f}"
    )
    ax.set_xlabel("Step inside window")
    ax.set_ylabel("z-normalized value")
    ax.legend(loc="upper right")
    fig.tight_layout()
    save_and_show_figure(fig, save_path)


if motifs_df.empty:
    print("Skipping intuitive plots because no primary motifs were extracted.")
else:
    display_value_col = "close" if "close" in df.columns else primary_result["feature"]
    plot_detected_motif_windows(
        df_signal=df[["timestamp", display_value_col]].dropna().copy(),
        value_col=display_value_col,
        motif_table=motifs_df,
        study_label=f"primary feature={primary_result['feature']} shown on {display_value_col}",
        save_path=build_plot_path(primary_result["feature"], "motif_windows"),
    )

    plot_matrix_profile_curve(
        result=primary_result,
        motif_table=motifs_df,
        save_path=build_plot_path(primary_result["feature"], "mp_plot"),
    )

    for _, motif_row in motifs_df.head(3).iterrows():
        plot_normalized_motif_overlay(
            result=primary_result,
            motif_row=motif_row,
            save_path=build_plot_path(primary_result["feature"], f"overlay_rank{int(motif_row['rank'])}"),
        )


## Optional Cached MSTUMP

The multivariate study is kept for thesis continuity, but it is guarded by the run-mode
configuration. Quick mode skips this section by default.


In [ ]:
if CFG["run_multivariate"]:
    selected_multi_features = [feature for feature in ["log_return", "rolling_volatility", "hl_range", "volume_zscore"] if feature in df.columns]
    print("Requested multivariate features:", selected_multi_features)

    if len(selected_multi_features) < 2:
        print("Skipping multivariate MSTUMP because fewer than two usable features are available.")
    else:
        if RUN_MODE == "full":
            print("Full-mode MSTUMP is CPU-heavy. This path is mainly intended for cached reruns or long execution sessions.")

        multivariate_result = compute_cached_mstump(
            df_in=df,
            features=selected_multi_features,
            m=CFG["m_intraday"],
            asset=ASSET,
            run_mode=RUN_MODE,
            force_recompute=FORCE_RECOMPUTE,
        )

        multivariate_motifs_df = extract_top_k_motifs(
            mp=multivariate_result["full_profile"],
            mpi=multivariate_result["full_indices"],
            timestamps=multivariate_result["timestamps"],
            m=multivariate_result["m"],
            top_k=CFG["top_k"],
        )
        multivariate_result["motifs_df"] = multivariate_motifs_df

        runtime_rows.append({
            "run_mode": RUN_MODE,
            "rows_used": int(multivariate_result["X"].shape[1]),
            "feature": "multivariate(" + ",".join(selected_multi_features) + ")",
            "window_size": int(multivariate_result["m"]),
            "top_k": int(CFG["top_k"]),
            "cache_used": bool(multivariate_result["cache_used"]),
            "elapsed_seconds": round(float(multivariate_result["elapsed_seconds"]), 4),
        })

        print("Multivariate motif table:")
        display(multivariate_motifs_df)
else:
    print("Multivariate MSTUMP skipped by configuration for this run mode.")


## Optional Event Matching

Event matching remains available for the thesis workflow, but it is disabled in quick mode so
the development notebook stays lightweight.


In [ ]:
EVENT_TOLERANCE_HOURS = 24


def load_external_events() -> pd.DataFrame:
    existing_paths = [path for path in EXTERNAL_EVENT_CANDIDATES if path.exists()]
    if existing_paths:
        event_path = existing_paths[0]
        print("Using external events file:", event_path)
        events = pd.read_csv(event_path)
    else:
        print("No external event CSV found. Using a small placeholder table for development.")
        events = pd.DataFrame([
            {
                "event_time": "2025-01-10 00:00:00+00:00",
                "event_name": "Placeholder: market stress event",
                "category": "placeholder",
            },
            {
                "event_time": "2025-03-14 00:00:00+00:00",
                "event_name": "Placeholder: sharp BTC move",
                "category": "placeholder",
            },
        ])

    required_columns = {"event_time", "event_name", "category"}
    missing_columns = sorted(required_columns - set(events.columns))
    if missing_columns:
        raise ValueError(f"External events file is missing columns: {missing_columns}")

    events = events.copy()
    events["event_time"] = pd.to_datetime(events["event_time"], utc=True, errors="coerce")
    events = events.dropna(subset=["event_time"]).sort_values("event_time").reset_index(drop=True)
    return events


def match_motifs_to_events(
    motif_occurrences: pd.DataFrame,
    events: pd.DataFrame,
    tolerance_hours: int = 24,
) -> pd.DataFrame:
    if motif_occurrences.empty or events.empty:
        return pd.DataFrame()

    tolerance = pd.Timedelta(hours=tolerance_hours)
    matches = []

    for _, motif_row in motif_occurrences.iterrows():
        for _, event_row in events.iterrows():
            gap = abs(motif_row["start"] - event_row["event_time"])
            if gap <= tolerance:
                matches.append({
                    "representation": motif_row["representation"],
                    "rank": int(motif_row["rank"]),
                    "occurrence_type": motif_row["occurrence_type"],
                    "window_size": int(motif_row["window_size"]),
                    "motif_start": motif_row["start"],
                    "event_time": event_row["event_time"],
                    "event_name": event_row["event_name"],
                    "event_category": event_row["category"],
                    "gap_hours": round(gap.total_seconds() / 3600, 3),
                })

    if not matches:
        return pd.DataFrame(columns=[
            "representation",
            "rank",
            "occurrence_type",
            "window_size",
            "motif_start",
            "event_time",
            "event_name",
            "event_category",
            "gap_hours",
        ])

    return pd.DataFrame(matches).sort_values("gap_hours").reset_index(drop=True)


if CFG["run_event_matching"]:
    occurrence_frames = []
    for feature_name, result in analysis_results.items():
        occurrence_frames.append(build_occurrence_table(result.get("motifs_df", empty_motif_table()), representation=feature_name))

    if multivariate_result is not None:
        occurrence_frames.append(
            build_occurrence_table(
                multivariate_result.get("motifs_df", empty_motif_table()),
                representation="multivariate",
            )
        )

    motif_occurrences = pd.concat([frame for frame in occurrence_frames if not frame.empty], ignore_index=True) if occurrence_frames else pd.DataFrame()
    events_df = load_external_events()
    event_matches = match_motifs_to_events(
        motif_occurrences=motif_occurrences,
        events=events_df,
        tolerance_hours=EVENT_TOLERANCE_HOURS,
    )
    display(event_matches.head(20))
else:
    print("Event matching skipped by configuration for this run mode.")


## Result Saving

The short notebook exports the main motif table, a run summary JSON, a runtime summary table,
and the lightweight PNG plots used in the discussion workflow.


In [ ]:
runtime_summary_df = pd.DataFrame(runtime_rows, columns=[
    "run_mode",
    "rows_used",
    "feature",
    "window_size",
    "top_k",
    "cache_used",
    "elapsed_seconds",
])

display(runtime_summary_df)

primary_feature_name = primary_result["feature"]

primary_motifs_path = RESULTS_DIR / f"{safe_stem(ASSET)}_{safe_stem(primary_feature_name)}_motifs_{RUN_MODE}.csv"
motifs_df.to_csv(primary_motifs_path, index=False)

runtime_summary_path = RESULTS_DIR / f"{safe_stem(ASSET)}_{safe_stem(primary_feature_name)}_runtime_summary_{RUN_MODE}.csv"
runtime_summary_df.to_csv(runtime_summary_path, index=False)

additional_outputs = {}

if raw_close_result is not None:
    raw_close_motifs_path = RESULTS_DIR / f"{safe_stem(ASSET)}_close_motifs_{RUN_MODE}.csv"
    raw_close_result["motifs_df"].to_csv(raw_close_motifs_path, index=False)
    additional_outputs["raw_close_motifs"] = str(raw_close_motifs_path)

if multivariate_result is not None:
    multivariate_motifs_path = RESULTS_DIR / f"{safe_stem(ASSET)}_multivariate_motifs_{RUN_MODE}.csv"
    multivariate_motifs_df.to_csv(multivariate_motifs_path, index=False)
    additional_outputs["multivariate_motifs"] = str(multivariate_motifs_path)

if not event_matches.empty:
    event_matches_path = RESULTS_DIR / f"{safe_stem(ASSET)}_event_matches_{RUN_MODE}.csv"
    event_matches.to_csv(event_matches_path, index=False)
    additional_outputs["event_matches"] = str(event_matches_path)

run_summary = {
    "asset": ASSET,
    "run_mode": RUN_MODE,
    "feature": primary_feature_name,
    "window_size": int(primary_result["m"]),
    "top_k": int(CFG["top_k"]),
    "rows_used": int(len(df)),
    "slice_start": str(df["timestamp"].min()),
    "slice_end": str(df["timestamp"].max()),
    "cache_used": bool(primary_result["cache_used"]),
    "elapsed_seconds": round(float(primary_result["elapsed_seconds"]), 6),
    "data_path": str(data_path),
    "cache_path": str(primary_result["cache_path"]),
    "plots_saved": [str(path) for path in saved_plot_paths],
    "additional_outputs": additional_outputs,
}

run_summary_path = RESULTS_DIR / f"{safe_stem(ASSET)}_{safe_stem(primary_feature_name)}_run_summary_{RUN_MODE}.json"
with run_summary_path.open("w", encoding="utf-8") as handle:
    json.dump(run_summary, handle, indent=2, default=str)

print("Saved primary motifs:", primary_motifs_path)
print("Saved runtime summary table:", runtime_summary_path)
print("Saved run summary JSON:", run_summary_path)
if saved_plot_paths:
    print("Saved plot files:")
    for plot_path in saved_plot_paths:
        print("-", plot_path.name)

print("\nFinal quick-study summary")
print("-------------------------")
print(f"run_mode      : {RUN_MODE}")
print(f"rows_used     : {len(df):,}")
print(f"feature       : {primary_feature_name}")
print(f"window_size   : {primary_result['m']}")
print(f"top_k         : {CFG['top_k']}")
print(f"cache_used    : {primary_result['cache_used']}")
print(f"elapsed_sec   : {primary_result['elapsed_seconds']:.4f}")


## Final Summary

This notebook is now the fast development version of the Matrix Profile study:

- `quick` is intended for fast motif checks and supervisor discussion
- `medium` expands the slice and can also run cached multivariate analysis
- `full` keeps the full-history thesis path, but it is best used with caches already built

The main output for discussion is the primary motif table together with the highlighted windows,
the Matrix Profile curve, and the normalized overlay plots.
